<a href="https://colab.research.google.com/github/michaelborck-curtin/smart-finance-assistant/blob/main/starter_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Project Overview

This notebook implements **Budget Buddy**, a Smart Finance Assistant that analyses transaction data and gives practical budgeting guidance.

**Final Application Components:**
- 💬 **AI Chat Interface** – a hands-on-ai powered finance chatbot using transaction context
- 📊 **Data Analysis** – CSV transaction loading, cleaning, and spending analysis
- 🔍 **CSV-Based RAG System** – retrieval of relevant transactions from the CSV based on the user's question
- 🛠️ **Custom Tool** – a savings goal calculator
- 🌐 **Gradio UI** – an interactive interface connecting the assistant components
- 🧪 **Testing** – function and integration tests for normal and invalid inputs

**Development Approach:** The project follows the six-step methodology: understand the problem, identify inputs and outputs, work by hand, write pseudocode, convert to Python, and test with different cases.

---

# 🚀 Getting Started: Foundation Setup

## Initial Setup
This cell installs the necessary libraries. In a Colab environment, you would uncomment the first line.

In [61]:
# Initial Setup
# Import the main libraries used throughout the Smart Finance Assistant

import warnings
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import gradio as gr

warnings.filterwarnings("ignore")

print("Core libraries loaded successfully.")
print(f"Pandas version: {pd.__version__}")

Core libraries loaded successfully.
Pandas version: 3.0.3


## Hands-on-AI Configuration

Set up the hands-on-ai package for advanced features (chat, RAG, tools):

In [62]:
# hands-on-ai External Server Setup
# Configuration based on the Assessment 2 unit announcement.

import os

os.environ["HANDS_ON_AI_SERVER"] = "https://ollama.locollm.org"
os.environ["HANDS_ON_AI_MODEL"] = "gemma3:4b"
os.environ["HANDS_ON_AI_API_KEY"] = "Curtin2026ISYS20015002"

print("Hands-on-AI configuration:")
print("Server:", os.environ["HANDS_ON_AI_SERVER"])
print("Model:", os.environ["HANDS_ON_AI_MODEL"])
print("API key length:", len(os.environ["HANDS_ON_AI_API_KEY"]))

Hands-on-AI configuration:
Server: https://ollama.locollm.org
Model: gemma3:4b
API key length: 22


## Connection Test

Test that everything is working correctly:

In [63]:
# Test the connection to the hands-on-ai server

from hands_on_ai.chat import get_response

response = get_response("Hello! Reply with only: server working")

print("Raw response:")
print(response)

if "401" in str(response) or "Authorization Required" in str(response) or "Error" in str(response):
    print("❌ Hands-on-AI server test failed.")
    print("The server rejected the request. Check the API key, server URL, model name, or package version.")
else:
    print("✅ Hands-on-AI server test successful.")

Raw response:
server working

✅ Hands-on-AI server test successful.


## Transaction CSV Setup

This project uses the provided `sample_transactions.csv` file as the working transaction dataset. The cell below checks that the file is available, displays the number of rows, confirms the column names, and previews the first few transactions.

In [65]:

# Inspect the actual transaction CSV used by the project

csv_files = ["sample_transactions.csv"]

for file_name in csv_files:
    if os.path.exists(file_name):
        print("=" * 60)
        print(f"File: {file_name}")
        data = pd.read_csv(file_name)
        print("Rows:", len(data))
        print("Columns:", data.columns.tolist())
        display(data.head())
    else:
        print(f"{file_name} is not in this folder.")

File: sample_transactions.csv
Rows: 25
Columns: ['Date', 'Amount', 'Category', 'Description']


,Date,Amount,Category,Description
0,2024-08-01,$45.50,Groceries,Woolworths Weekly Shop
1,2024-08-02,$12.00,Transport,Opal Card Top-up
2,2024-08-03,$89.95,Entertainment,Concert Tickets - Enmore Theatre
3,2024-08-04,$3.50,Coffee,Campus Cafe Flat White
4,2024-08-05,$120.00,Groceries,Coles Weekly Shop


# 📊 Six-Step Development Methodology

This section explains the development process used for the Budget Buddy Smart Finance Assistant. The six-step method is applied to the whole project rather than repeated separately for every function. The Python implementation cells that follow this section represent Step 5, and the testing section near the end of the notebook represents Step 6.

---

## STEP 1: Understand the Problem

Many students and young adults have transaction records from bank accounts or budgeting apps, but they do not always know how to interpret the data. A list of transactions can show what was purchased, but it does not clearly explain where money is going, which categories matter most, or what practical action should be taken.

This project builds **Budget Buddy**, a Smart Finance Assistant that helps users understand spending behaviour from a transaction CSV file. The assistant loads transaction data, cleans it, analyses spending by category, identifies refunds, highlights the largest spending areas, and gives practical budgeting advice through a chatbot-style interface.

The business value of the project is that it turns raw transaction data into clear finance insights. Instead of only showing totals, the assistant explains what the figures mean and gives realistic suggestions such as reviewing grocery spending, setting weekly limits, and calculating savings goals.

---

## STEP 2: Identify Inputs and Outputs

### Inputs

The assistant uses:

- a transaction CSV file with `Date`, `Amount`, `Category`, and `Description` columns;
- a monthly income value entered by the user;
- a finance question entered by the user;
- savings goal inputs: goal amount, current savings, and monthly contribution.

Example user questions include:

```text
How can I reduce my grocery spending?
What is my largest spending category?
How much am I saving?
```

### Outputs

The assistant produces:

- cleaned transaction data;
- total number of transactions;
- number of expense and refund transactions;
- net expenses after refund handling;
- spending by category;
- largest spending category;
- average expense transaction;
- estimated savings and savings rate based on monthly income;
- relevant transactions retrieved from the CSV based on the user's question;
- a chatbot response using the user's transaction data;
- a savings goal calculation;
- a Gradio interface connecting the analysis, chatbot, and savings tool.

---

## STEP 3: Work the Problem by Hand

Before coding, the main calculations were checked manually using the sample transaction data.

### Manual Example 1: Grocery Spending

The sample data includes these grocery transactions:

| Description | Amount |
|---|---:|
| Woolworths Weekly Shop | $45.50 |
| Coles Weekly Shop | $120.00 |
| Woolworths Fortnightly Shop | $156.00 |
| IGA Local Shop | $73.20 |
| Monthly Bulk Shopping - Costco | $145.60 |

Manual calculation:

```text
Grocery spending = 45.50 + 120.00 + 156.00 + 73.20 + 145.60
Grocery spending = 540.30
```

This confirms why Groceries should appear as the largest spending category.

### Manual Example 2: Refund Handling

The sample data contains refund transactions. Because the CSV stores all amounts as positive values, refunds should not be treated as normal expenses.

Manual logic:

```text
Raw total including refund rows = $1246.15
Refunds identified = $37.50
Normal expenses excluding refund rows = $1208.65
Net expenses used for analysis = $1208.65
```

The refund rows are reported separately and excluded from spending categories. They are not subtracted again after being excluded, because that would double-count the refund adjustment.

### Manual Example 3: Estimated Savings Rate

Using a monthly income of $2000:

```text
Estimated savings = Monthly income - Net expenses
Estimated savings = 2000.00 - 1208.65
Estimated savings = 791.35
```

Savings rate:

```text
Savings rate = Estimated savings / Monthly income × 100
Savings rate = 791.35 / 2000.00 × 100
Savings rate = 39.57%
```

This shows that the user has an estimated positive savings position, but the assistant can still help improve the largest spending category.

---

## STEP 4: Write Pseudocode

```text
START Budget Buddy Smart Finance Assistant

LOAD transaction CSV file

CHECK that required columns exist:
    Date
    Amount
    Category
    Description

CLEAN the data:
    Convert Date column into date format
    Remove dollar signs, commas, and spaces from Amount
    Convert Amount into numeric values
    Remove invalid rows
    Standardise Category and Description text

CLASSIFY transactions:
    Identify income transactions using keywords
    Identify refund transactions using keywords
    Treat remaining transactions as expenses

ANALYSE spending:
    Calculate total expenses
    Calculate total refunds
    Calculate average expense transaction
    Group expenses by category
    Find the largest spending category

GENERATE business insights:
    Show net expenses
    Show refund amount
    Show largest category
    If monthly income is provided:
        Calculate estimated savings
        Calculate savings rate
        Give savings interpretation

RETRIEVE relevant CSV context:
    Read user's question
    Match question keywords against Category and Description
    Return relevant transactions
    If no match exists, return largest transactions

CREATE chatbot response:
    Use the user's question
    Use spending summary
    Use retrieved transaction context
    Provide warm and practical finance guidance

RUN custom savings goal tool:
    Ask for goal amount
    Ask for current savings
    Ask for monthly contribution
    Calculate remaining amount
    Calculate months needed

BUILD Gradio interface:
    Allow optional CSV upload
    Allow monthly income entry
    Allow finance question entry
    Allow savings goal inputs
    Display analysis, chatbot response, and savings goal output

TEST the project:
    Test valid CSV
    Test missing file
    Test missing columns
    Test invalid data
    Test spending calculations
    Test RAG context
    Test chatbot response
    Test savings calculator

END
```

---

## STEP 5: Convert to Python

The following Python cells implement the pseudocode above. The implementation is split into clear sections:

- initial setup and imports;
- CSV inspection;
- data loading and cleaning;
- spending analysis;
- business insight generation;
- chatbot integration;
- CSV-based RAG/context retrieval;
- custom savings goal tool;
- Gradio interface;
- backend and interface tests.

AI assistance was used to help draft and refine parts of the code, but the outputs were tested in the notebook and adjusted when the financial logic needed improvement, such as the refund-handling calculation.

---

## STEP 6: Test with a Variety of Data

The final testing section near the end of this notebook tests the completed project. It checks normal cases, expected calculations, chatbot behaviour, RAG output, savings calculator output, and error handling for bad inputs such as missing files, missing columns, and invalid amount data.

---

# 🧹 Data Loading and Cleaning

This section loads the transaction CSV file and prepares it for analysis. The cleaning function checks that the file contains the required columns, converts dates into a usable format, converts the `Amount` column into numeric values, removes invalid rows, and standardises text columns.


In [66]:
def load_and_clean_transaction_data(file_path):
    """
    Load and clean transaction data for the Budget Buddy Smart Finance Assistant.

    Expected CSV columns:
    - Date
    - Amount
    - Category
    - Description

    Args:
        file_path: Path to the transaction CSV file.

    Returns:
        pandas.DataFrame or None: Cleaned transaction data if successful.
    """
    required_columns = ["Date", "Amount", "Category", "Description"]

    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print("Error: The transaction file could not be found.")
        return None
    except Exception as error:
        print(f"Error loading file: {error}")
        return None

    # Check that the CSV has the required columns
    missing_columns = []

    for column in required_columns:
        if column not in df.columns:
            missing_columns.append(column)

    if len(missing_columns) > 0:
        print(f"Error: Missing required columns: {missing_columns}")
        return None

    # Work on a copy so the original data is not changed accidentally
    df = df.copy()

    # Convert Date into datetime format
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Clean Amount in case it contains dollar signs, commas, or spaces
    df["Amount"] = (
        df["Amount"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

    # Remove rows where important data could not be read
    df = df.dropna(subset=["Date", "Amount", "Category", "Description"])

    # Clean text columns
    df["Category"] = df["Category"].astype(str).str.strip()
    df["Description"] = df["Description"].astype(str).str.strip()

    if len(df) == 0:
        print("Error: No valid transactions were found after cleaning.")
        return None

    print("Transaction data loaded and cleaned successfully.")
    print(f"Rows loaded: {len(df)}")

    return df

In [67]:
clean_data = load_and_clean_transaction_data("sample_transactions.csv")

if clean_data is not None:
    display(clean_data.head())

Transaction data loaded and cleaned successfully.
Rows loaded: 25


,Date,Amount,Category,Description
0,2024-08-01,45.50,Groceries,Woolworths Weekly Shop
1,2024-08-02,12.00,Transport,Opal Card Top-up
2,2024-08-03,89.95,Entertainment,Concert Tickets - Enmore Theatre
3,2024-08-04,3.50,Coffee,Campus Cafe Flat White
4,2024-08-05,120.00,Groceries,Coles Weekly Shop


## 📈 Spending Pattern Analysis

This function separates expense and refund transactions, calculates net expenses, groups spending by category, and identifies the largest spending category.

In [68]:
# Spending Analysis Function
# This function turns cleaned transaction data into useful finance insights.

def analyze_spending_patterns(df):
    """
    Analyse spending patterns from cleaned transaction data.

    The sample transaction file stores amounts as positive numbers.
    This function treats normal transactions as expenses, treats refunds
    as expense reductions, and identifies income if income keywords are present.

    Args:
        df: Cleaned transaction DataFrame

    Returns:
        dict: Spending summary and category-level analysis
    """
    if df is None:
        return None

    if len(df) == 0:
        return None

    data = df.copy()

    # Create lowercase helper columns for simple classification
    data["category_lower"] = data["Category"].astype(str).str.lower()
    data["description_lower"] = data["Description"].astype(str).str.lower()

    income_keywords = ["income", "salary", "wage", "pay", "allowance", "deposit"]
    refund_keywords = ["refund", "reimbursement", "cashback"]

    def contains_keyword(text, keywords):
        for keyword in keywords:
            if keyword in text:
                return True
        return False

    def is_income_transaction(row):
        combined_text = row["category_lower"] + " " + row["description_lower"]
        return contains_keyword(combined_text, income_keywords)

    def is_refund_transaction(row):
        combined_text = row["category_lower"] + " " + row["description_lower"]
        return contains_keyword(combined_text, refund_keywords)

    data["Is_Income"] = data.apply(is_income_transaction, axis=1)
    data["Is_Refund"] = data.apply(is_refund_transaction, axis=1)

    income_data = data[data["Is_Income"] == True]
    refund_data = data[data["Is_Refund"] == True]
    expense_data = data[(data["Is_Income"] == False) & (data["Is_Refund"] == False)]

    total_income = income_data["Amount"].abs().sum()
    gross_expenses = expense_data["Amount"].abs().sum()
    total_refunds = refund_data["Amount"].abs().sum()

    # Refunds are excluded from expenses and reported separately.
    # Because the sample file stores all amounts as positive values,
    # subtracting refunds again would double-count the adjustment.
    total_expenses = gross_expenses

    net_savings = total_income - total_expenses

    if total_income > 0:
        savings_rate = (net_savings / total_income) * 100
    else:
        savings_rate = 0

    category_spending = (
        expense_data
        .groupby("Category")["Amount"]
        .sum()
        .abs()
        .sort_values(ascending=False)
    )

    if len(category_spending) > 0:
        highest_category = category_spending.index[0]
        highest_category_amount = category_spending.iloc[0]
    else:
        highest_category = "No expense categories found"
        highest_category_amount = 0

    if len(expense_data) > 0:
        average_transaction = expense_data["Amount"].abs().mean()
    else:
        average_transaction = 0

    summary = {
        "total_income": total_income,
        "gross_expenses": gross_expenses,
        "total_refunds": total_refunds,
        "total_expenses": total_expenses,
        "net_savings": net_savings,
        "savings_rate": savings_rate,
        "category_spending": category_spending,
        "highest_category": highest_category,
        "highest_category_amount": highest_category_amount,
        "average_transaction": average_transaction,
        "number_of_transactions": len(data),
        "number_of_income_transactions": len(income_data),
        "number_of_refund_transactions": len(refund_data),
        "number_of_expense_transactions": len(expense_data)
    }

    return summary

In [69]:
# Test the spending analysis function

spending_summary = analyze_spending_patterns(clean_data)

print("Finance Summary")
print("-" * 40)
print(f"Number of transactions: {spending_summary['number_of_transactions']}")
print(f"Income transactions: {spending_summary['number_of_income_transactions']}")
print(f"Refund transactions: {spending_summary['number_of_refund_transactions']}")
print(f"Expense transactions: {spending_summary['number_of_expense_transactions']}")
print(f"Total income: ${spending_summary['total_income']:.2f}")
print(f"Gross expenses before refunds: ${spending_summary['gross_expenses']:.2f}")
print(f"Total refunds: ${spending_summary['total_refunds']:.2f}")
print(f"Net expenses after refunds: ${spending_summary['total_expenses']:.2f}")
print(f"Net savings: ${spending_summary['net_savings']:.2f}")
print(f"Savings rate: {spending_summary['savings_rate']:.2f}%")
print(f"Highest spending category: {spending_summary['highest_category']} (${spending_summary['highest_category_amount']:.2f})")
print(f"Average expense transaction: ${spending_summary['average_transaction']:.2f}")

print("\nSpending by category:")
display(spending_summary["category_spending"])

Finance Summary
----------------------------------------
Number of transactions: 25
Income transactions: 0
Refund transactions: 2
Expense transactions: 23
Total income: $0.00
Gross expenses before refunds: $1208.65
Total refunds: $37.50
Net expenses after refunds: $1208.65
Net savings: $-1208.65
Savings rate: 0.00%
Highest spending category: Groceries ($540.30)
Average expense transaction: $52.55

Spending by category:


Category
Groceries        540.30
Entertainment    206.95
Utilities        159.80
Dining           148.85
Transport        108.00
Coffee            44.75
Name: Amount, dtype: float64

## 💡 Business Insights Generator

This function converts the spending summary into plain-English finance insights, including estimated savings and savings rate when monthly income is provided.

In [70]:
# Business Insights Generator
# This function converts the numerical spending summary into plain-English advice.

def generate_business_insights(spending_summary, monthly_income=0):
    """
    Generate practical finance insights from the spending analysis.

    Args:
        spending_summary: Dictionary returned by analyze_spending_patterns()
        monthly_income: Optional monthly income entered by the user

    Returns:
        str: Plain-English finance insights
    """
    if spending_summary is None:
        return "No spending summary is available. Please load and analyse transaction data first."

    total_expenses = spending_summary["total_expenses"]
    highest_category = spending_summary["highest_category"]
    highest_category_amount = spending_summary["highest_category_amount"]
    average_transaction = spending_summary["average_transaction"]
    total_refunds = spending_summary["total_refunds"]
    category_spending = spending_summary["category_spending"]

    insights = []

    insights.append("Budget Buddy Finance Insights")
    insights.append("-" * 40)

    insights.append(f"Net expenses after refunds: ${total_expenses:.2f}")
    insights.append(f"Refunds identified: ${total_refunds:.2f}")
    insights.append(f"Average expense transaction: ${average_transaction:.2f}")
    insights.append(f"Largest spending category: {highest_category} (${highest_category_amount:.2f})")

    if monthly_income > 0:
        estimated_savings = monthly_income - total_expenses
        savings_rate = (estimated_savings / monthly_income) * 100

        insights.append("")
        insights.append(f"Monthly income entered: ${monthly_income:.2f}")
        insights.append(f"Estimated savings after expenses: ${estimated_savings:.2f}")
        insights.append(f"Estimated savings rate: {savings_rate:.2f}%")

        if savings_rate >= 20:
            insights.append("Your estimated savings rate is strong. You may have room to build savings or invest after keeping an emergency fund.")
        elif savings_rate >= 10:
            insights.append("Your estimated savings rate is reasonable, but there may still be room to reduce flexible spending.")
        elif savings_rate > 0:
            insights.append("Your estimated savings rate is low. Review high-spending categories and set a realistic spending limit.")
        else:
            insights.append("Your expenses appear to exceed your income. Reducing non-essential spending should be the first priority.")
    else:
        insights.append("")
        insights.append("No monthly income was provided, so savings rate cannot be calculated accurately.")
        insights.append("The analysis focuses on spending behaviour only.")

    insights.append("")
    insights.append("Category-level observations:")

    for category, amount in category_spending.items():
        category_share = (amount / total_expenses) * 100 if total_expenses > 0 else 0
        insights.append(f"- {category}: ${amount:.2f}, which is {category_share:.1f}% of net expenses.")

    insights.append("")
    insights.append("Suggested action:")
    insights.append(f"Start by reviewing {highest_category}, because it is the largest spending category in this dataset.")

    return "\n".join(insights)

In [71]:
# Test the business insights generator

insight_text = generate_business_insights(spending_summary, monthly_income=2000)

print(insight_text)

Budget Buddy Finance Insights
----------------------------------------
Net expenses after refunds: $1208.65
Refunds identified: $37.50
Average expense transaction: $52.55
Largest spending category: Groceries ($540.30)

Monthly income entered: $2000.00
Estimated savings after expenses: $791.35
Estimated savings rate: 39.57%
Your estimated savings rate is strong. You may have room to build savings or invest after keeping an emergency fund.

Category-level observations:
- Groceries: $540.30, which is 44.7% of net expenses.
- Entertainment: $206.95, which is 17.1% of net expenses.
- Utilities: $159.80, which is 13.2% of net expenses.
- Dining: $148.85, which is 12.3% of net expenses.
- Transport: $108.00, which is 8.9% of net expenses.
- Coffee: $44.75, which is 3.7% of net expenses.

Suggested action:
Start by reviewing Groceries, because it is the largest spending category in this dataset.


---

# 🌐 Advanced Features: Integrating AI Components

This section integrates the advanced AI components of Budget Buddy, including the hands-on-ai chat response, CSV-based RAG context, custom savings tool, and Gradio interface.

## Chat Interface Integration

In [72]:
# Chat Interface Integration
# This chatbot uses hands-on-ai external chat plus CSV/RAG transaction context.

from hands_on_ai.chat import get_response

def budget_buddy_chatbot(user_question, df, spending_summary, monthly_income=0):
    """
    Generate a finance-focused chatbot response using hands-on-ai.

    The chatbot receives:
    - the user's finance question
    - the transaction summary
    - the CSV/RAG context
    - instructions to answer warmly and practically

    Args:
        user_question: Question typed by the user
        df: Cleaned transaction DataFrame
        spending_summary: Dictionary from analyze_spending_patterns()
        monthly_income: Optional monthly income entered by the user

    Returns:
        str: Chatbot response from hands-on-ai
    """
    if user_question is None or str(user_question).strip() == "":
        return "Please ask a finance question about your spending, budget, or savings goal."

    if df is None or spending_summary is None:
        return "Please load and analyse transaction data before asking a finance question."

    rag_context = create_transaction_context(
        df,
        spending_summary,
        user_question=user_question,
        monthly_income=monthly_income
    )

    prompt = f"""
You are Budget Buddy, a warm and practical personal finance assistant for students and young adults.

Use ONLY the transaction data and summary provided below. Do not invent extra transactions or figures.

Your response should:
- directly answer the user's question;
- use the actual figures from the transaction data;
- sound encouraging and supportive;
- give realistic practical actions;
- avoid being too vague;
- avoid sounding too formal;
- mention that this is general budgeting guidance, not professional financial advice.

User question:
{user_question}

Transaction and finance context:
{rag_context}

Write a helpful response in 3 to 6 short paragraphs or bullet points.
"""

    ai_response = get_response(prompt)

    response_lines = []
    response_lines.append("Budget Buddy Response")
    response_lines.append("-" * 40)
    response_lines.append(ai_response)
    response_lines.append("")
    response_lines.append("Relevant data used:")
    response_lines.append(rag_context)

    return "\n".join(response_lines)

## CSV-Based RAG System

This section retrieves relevant transaction rows from the CSV file based on the user's finance question. The retrieved rows are added to the chatbot prompt so the hands-on-ai response is grounded in the user's transaction data.

In [73]:
# RAG / CSV Context Retrieval
# This section creates finance context from the transaction CSV so the chatbot can answer using the user's data.

def format_currency(amount):
    """
    Format a number as currency for readable outputs.
    """
    return f"${amount:,.2f}"


def retrieve_relevant_transactions(df, user_question, max_rows=5):
    """
    Retrieve transactions that are relevant to a user's question.

    This is a simple CSV-based retrieval method:
    - It checks whether words from the user's question appear in the Category or Description.
    - It returns the most relevant matching rows.
    - If there are no matches, it returns the largest transactions instead.

    Args:
        df: Cleaned transaction DataFrame
        user_question: User's finance question
        max_rows: Maximum number of rows to return

    Returns:
        pandas.DataFrame: Relevant transaction rows
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()

    data = df.copy()

    question_words = (
        str(user_question)
        .lower()
        .replace("?", "")
        .replace(",", "")
        .replace(".", "")
        .split()
    )

    # Remove very common words that do not help retrieval
    stop_words = [
        "what", "why", "how", "can", "could", "should", "would",
        "the", "a", "an", "my", "i", "me", "to", "for", "of",
        "in", "on", "is", "are", "do", "about", "spending", "spend",
        "reduce", "lower", "cut", "save"
    ]

    search_words = []

    for word in question_words:
        if word not in stop_words and len(word) > 2:
            search_words.append(word)

            # Add a basic plural/singular version to catch grocery/groceries
            if word.endswith("ies"):
                search_words.append(word[:-3] + "y")
            elif word.endswith("y"):
                search_words.append(word[:-1] + "ies")
            elif word.endswith("s"):
                search_words.append(word[:-1])
            else:
                search_words.append(word + "s")

    data["search_text"] = (
        data["Category"].astype(str).str.lower()
        + " "
        + data["Description"].astype(str).str.lower()
    )

    if len(search_words) > 0:
        match_mask = data["search_text"].apply(
            lambda text: any(word in text for word in search_words)
        )

        matches = data[match_mask].copy()

        if len(matches) > 0:
            return matches.head(max_rows)

    # Fallback: return largest transactions if no specific keyword match is found
    return data.sort_values("Amount", ascending=False).head(max_rows)


def create_transaction_context(df, spending_summary, user_question="", monthly_income=0):
    """
    Create a text context from CSV transaction data.

    This context can be passed into a chatbot prompt so that the chatbot
    answers using the user's transaction data.

    Args:
        df: Cleaned transaction DataFrame
        spending_summary: Dictionary returned by analyze_spending_patterns()
        user_question: User's question
        monthly_income: Optional monthly income entered by the user

    Returns:
        str: Context text for chatbot/RAG use
    """
    if df is None or spending_summary is None:
        return "No transaction data is available."

    relevant_transactions = retrieve_relevant_transactions(df, user_question)

    context_lines = []

    context_lines.append("TRANSACTION DATA CONTEXT")
    context_lines.append("-" * 40)
    context_lines.append(f"Number of transactions: {spending_summary['number_of_transactions']}")
    context_lines.append(f"Expense transactions: {spending_summary['number_of_expense_transactions']}")
    context_lines.append(f"Refund transactions: {spending_summary['number_of_refund_transactions']}")
    context_lines.append(f"Net expenses: {format_currency(spending_summary['total_expenses'])}")
    context_lines.append(f"Refunds identified: {format_currency(spending_summary['total_refunds'])}")
    context_lines.append(f"Highest spending category: {spending_summary['highest_category']} ({format_currency(spending_summary['highest_category_amount'])})")

    if monthly_income > 0:
        estimated_savings = monthly_income - spending_summary["total_expenses"]
        savings_rate = (estimated_savings / monthly_income) * 100

        context_lines.append(f"Monthly income entered: {format_currency(monthly_income)}")
        context_lines.append(f"Estimated savings: {format_currency(estimated_savings)}")
        context_lines.append(f"Estimated savings rate: {savings_rate:.2f}%")
    else:
        context_lines.append("Monthly income was not provided.")

    context_lines.append("")
    context_lines.append("Spending by category:")

    for category, amount in spending_summary["category_spending"].items():
        context_lines.append(f"- {category}: {format_currency(amount)}")

    context_lines.append("")
    context_lines.append("Relevant transactions for the user's question:")

    if len(relevant_transactions) == 0:
        context_lines.append("No relevant transactions found.")
    else:
        for _, row in relevant_transactions.iterrows():
            transaction_date = row["Date"]

            if hasattr(transaction_date, "date"):
                transaction_date = transaction_date.date()

            context_lines.append(
                f"- {transaction_date} | {row['Category']} | {format_currency(row['Amount'])} | {row['Description']}"
            )

    return "\n".join(context_lines)

In [74]:
# Test the CSV-based RAG context

sample_question = "How can I reduce my grocery spending?"

rag_context = create_transaction_context(
    clean_data,
    spending_summary,
    user_question=sample_question,
    monthly_income=2000
)

print(rag_context)

TRANSACTION DATA CONTEXT
----------------------------------------
Number of transactions: 25
Expense transactions: 23
Refund transactions: 2
Net expenses: $1,208.65
Refunds identified: $37.50
Highest spending category: Groceries ($540.30)
Monthly income entered: $2,000.00
Estimated savings: $791.35
Estimated savings rate: 39.57%

Spending by category:
- Groceries: $540.30
- Entertainment: $206.95
- Utilities: $159.80
- Dining: $148.85
- Transport: $108.00
- Coffee: $44.75

Relevant transactions for the user's question:
- 2024-08-01 | Groceries | $45.50 | Woolworths Weekly Shop
- 2024-08-05 | Groceries | $120.00 | Coles Weekly Shop
- 2024-08-12 | Groceries | $156.00 | Woolworths Fortnightly Shop
- 2024-08-18 | Groceries | $73.20 | IGA Local Shop
- 2024-08-25 | Groceries | $145.60 | Monthly Bulk Shopping - Costco


In [75]:
# Test the Budget Buddy chatbot after RAG functions have been defined

chatbot_test_response = budget_buddy_chatbot(
    user_question="How can I reduce my grocery spending?",
    df=clean_data,
    spending_summary=spending_summary,
    monthly_income=2000
)

print(chatbot_test_response)

Budget Buddy Response
----------------------------------------
Hey there! Let’s tackle those grocery bills – you’ve got this! It’s awesome that you’re looking for ways to save, especially since groceries are your biggest expense at $540.30. That’s a really smart move to ask for help. 

Looking at your recent purchases, you’ve been doing a mix of shopping – from weekly trips to Woolworths and Coles, to a bulk Costco run. To really bring that number down, let’s focus on a few things. Firstly, try planning your meals for the week *before* you go shopping and stick to a list. That way, you're less likely to grab impulse items! 

Secondly, comparing prices between your different stores (Woolworths, Coles, IGA, Costco) can make a big difference. The Costco shop at $145.60 was great, but maybe we can focus on smaller, more frequent trips for essentials.  

It’s also worth noting that you spent $45.50 on a weekly shop at Woolworths – let’s see if we can keep those smaller trips consistent.  Wi

## Custom Financial Tools

In [76]:
# Custom Financial Tool
# This tool calculates how long it will take to reach a savings goal.

def calculate_savings_goal(goal_amount, current_savings, monthly_contribution):
    """
    Calculate how many months are needed to reach a savings goal.

    Args:
        goal_amount: Target savings amount
        current_savings: Current amount already saved
        monthly_contribution: Amount the user can save each month

    Returns:
        str: Savings goal result in plain English
    """
    try:
        goal_amount = float(goal_amount)
        current_savings = float(current_savings)
        monthly_contribution = float(monthly_contribution)
    except ValueError:
        return "Please enter valid numbers for the goal amount, current savings, and monthly contribution."

    if goal_amount <= 0:
        return "The goal amount must be greater than zero."

    if current_savings < 0:
        return "Current savings cannot be negative."

    if monthly_contribution <= 0:
        return "Monthly contribution must be greater than zero."

    remaining_amount = goal_amount - current_savings

    if remaining_amount <= 0:
        return "You have already reached this savings goal."

    months_needed = int(np.ceil(remaining_amount / monthly_contribution))
    years = months_needed // 12
    months = months_needed % 12

    result_lines = []

    result_lines.append("Savings Goal Calculator")
    result_lines.append("-" * 40)
    result_lines.append(f"Goal amount: ${goal_amount:,.2f}")
    result_lines.append(f"Current savings: ${current_savings:,.2f}")
    result_lines.append(f"Monthly contribution: ${monthly_contribution:,.2f}")
    result_lines.append(f"Remaining amount: ${remaining_amount:,.2f}")
    result_lines.append(f"Months needed: {months_needed}")

    if years > 0:
        result_lines.append(f"Approximate time: {years} year(s) and {months} month(s)")
    else:
        result_lines.append(f"Approximate time: {months} month(s)")

    return "\n".join(result_lines)

In [77]:
# Test the custom savings goal tool

savings_goal_result = calculate_savings_goal(
    goal_amount=5000,
    current_savings=1200,
    monthly_contribution=400
)

print(savings_goal_result)

Savings Goal Calculator
----------------------------------------
Goal amount: $5,000.00
Current savings: $1,200.00
Monthly contribution: $400.00
Remaining amount: $3,800.00
Months needed: 10
Approximate time: 10 month(s)


## Gradio UI Integration

In [78]:
# Gradio UI Integration
# This interface connects CSV analysis, RAG context, chatbot responses, and the savings goal tool.

def get_file_path(uploaded_file):
    """
    Get a usable file path from a Gradio file upload.

    If no file is uploaded, the app uses the default sample transaction file.
    """
    if uploaded_file is None:
        return "sample_transactions.csv"

    if isinstance(uploaded_file, str):
        return uploaded_file

    if hasattr(uploaded_file, "name"):
        return uploaded_file.name

    return uploaded_file


def run_budget_buddy_interface(
    uploaded_file,
    monthly_income,
    user_question,
    goal_amount,
    current_savings,
    monthly_contribution
):
    """
    Main function used by the Gradio interface.

    It:
    1. Loads and cleans the transaction CSV
    2. Analyses spending patterns
    3. Generates business insights
    4. Answers the user's question using chatbot + CSV context
    5. Runs the custom savings goal tool

    Returns:
        tuple: analysis output, chatbot output, savings tool output
    """
    file_path = get_file_path(uploaded_file)

    cleaned_transactions = load_and_clean_transaction_data(file_path)

    if cleaned_transactions is None:
        return (
            "The transaction file could not be loaded. Please check that it has Date, Amount, Category, and Description columns.",
            "No chatbot response is available because the data could not be loaded.",
            "No savings goal calculation is available because the data could not be loaded."
        )

    current_summary = analyze_spending_patterns(cleaned_transactions)

    if current_summary is None:
        return (
            "The transaction file was loaded, but no valid transactions could be analysed.",
            "No chatbot response is available because the analysis failed.",
            "No savings goal calculation is available because the analysis failed."
        )

    try:
        monthly_income = float(monthly_income)
    except ValueError:
        monthly_income = 0

    analysis_output = generate_business_insights(
        current_summary,
        monthly_income=monthly_income
    )

    chatbot_output = budget_buddy_chatbot(
        user_question=user_question,
        df=cleaned_transactions,
        spending_summary=current_summary,
        monthly_income=monthly_income
    )

    savings_tool_output = calculate_savings_goal(
        goal_amount=goal_amount,
        current_savings=current_savings,
        monthly_contribution=monthly_contribution
    )

    return analysis_output, chatbot_output, savings_tool_output


with gr.Blocks(title="Budget Buddy Smart Finance Assistant") as budget_buddy_app:
    gr.Markdown("# Budget Buddy Smart Finance Assistant")
    gr.Markdown(
        "Upload a transaction CSV or use the default sample file. "
        "The assistant analyses spending, answers finance questions, "
        "and calculates a savings goal."
    )

    with gr.Row():
        uploaded_file = gr.File(
            label="Upload transaction CSV (optional)",
            file_types=[".csv"]
        )

        monthly_income = gr.Number(
            label="Monthly income",
            value=2000
        )

    user_question = gr.Textbox(
        label="Ask a finance question",
        value="How can I reduce my grocery spending?",
        lines=2
    )

    gr.Markdown("## Savings Goal Tool")

    with gr.Row():
        goal_amount = gr.Number(
            label="Goal amount",
            value=5000
        )

        current_savings = gr.Number(
            label="Current savings",
            value=1200
        )

        monthly_contribution = gr.Number(
            label="Monthly contribution",
            value=400
        )

    run_button = gr.Button("Run Budget Buddy")

    analysis_output = gr.Textbox(
        label="Spending Analysis and Business Insights",
        lines=15
    )

    chatbot_output = gr.Textbox(
        label="Chatbot Response with CSV Context",
        lines=20
    )

    savings_tool_output = gr.Textbox(
        label="Savings Goal Tool Output",
        lines=10
    )

    run_button.click(
        fn=run_budget_buddy_interface,
        inputs=[
            uploaded_file,
            monthly_income,
            user_question,
            goal_amount,
            current_savings,
            monthly_contribution
        ],
        outputs=[
            analysis_output,
            chatbot_output,
            savings_tool_output
        ]
    )

In [79]:
# Test the Gradio backend function before launching the interface

test_analysis_output, test_chatbot_output, test_savings_output = run_budget_buddy_interface(
    uploaded_file=None,
    monthly_income=2000,
    user_question="How can I reduce my grocery spending?",
    goal_amount=5000,
    current_savings=1200,
    monthly_contribution=400
)

print("Analysis output preview:")
print(test_analysis_output[:500])

print("\nChatbot output preview:")
print(test_chatbot_output[:500])

print("\nSavings tool output:")
print(test_savings_output)

Transaction data loaded and cleaned successfully.
Rows loaded: 25
Analysis output preview:
Budget Buddy Finance Insights
----------------------------------------
Net expenses after refunds: $1208.65
Refunds identified: $37.50
Average expense transaction: $52.55
Largest spending category: Groceries ($540.30)

Monthly income entered: $2000.00
Estimated savings after expenses: $791.35
Estimated savings rate: 39.57%
Your estimated savings rate is strong. You may have room to build savings or invest after keeping an emergency fund.

Category-level observations:
- Groceries: $540.30, which 

Chatbot output preview:
Budget Buddy Response
----------------------------------------
Hey there! Let’s tackle those grocery bills – you’ve got this! It’s awesome that you’re looking for ways to save, and honestly, groceries are a really common area where people can make a big difference. Based on your spending, groceries are currently your biggest expense at $540.30, so focusing here will definitely hav

---

# 🧪 STEP 6: Test with a Variety of Data

In [ ]:
# These tests check normal cases, error cases, and edge cases.

import io

def run_project_tests():
    """
    Run simple tests for the Budget Buddy Smart Finance Assistant.
    These tests are written in plain Python so they are easy to read in the notebook.
    """
    print("Budget Buddy Test Results")
    print("-" * 40)

    tests_passed = 0
    tests_failed = 0

    def check_test(test_name, condition):
        nonlocal tests_passed, tests_failed

        if condition:
            print(f"PASS: {test_name}")
            tests_passed += 1
        else:
            print(f"FAIL: {test_name}")
            tests_failed += 1

    # Test 1: Valid CSV loads correctly
    valid_data = load_and_clean_transaction_data("sample_transactions.csv")
    check_test("Valid CSV loads successfully", valid_data is not None)
    check_test("Valid CSV has 25 rows", len(valid_data) == 25)

    # Test 2: Spending analysis returns expected key values
    valid_summary = analyze_spending_patterns(valid_data)
    check_test("Spending summary is created", valid_summary is not None)
    check_test("Groceries is the largest spending category", valid_summary["highest_category"] == "Groceries")
    check_test("Net expenses are calculated correctly", round(valid_summary["total_expenses"], 2) == 1208.65)

    # Test 3: Business insights return readable text
    insights = generate_business_insights(valid_summary, monthly_income=2000)
    check_test("Business insights include Budget Buddy heading", "Budget Buddy Finance Insights" in insights)
    check_test("Business insights include groceries", "Groceries" in insights)

    # Test 4: RAG context retrieves grocery transactions
    grocery_context = create_transaction_context(
        valid_data,
        valid_summary,
        user_question="How can I reduce grocery spending?",
        monthly_income=2000
    )
    check_test("RAG context includes transaction data heading", "TRANSACTION DATA CONTEXT" in grocery_context)
    check_test("RAG context includes grocery transactions", "Groceries" in grocery_context)

    # Test 5: Chatbot answers a grocery question
    chatbot_response = budget_buddy_chatbot(
        user_question="How can I reduce my grocery spending?",
        df=valid_data,
        spending_summary=valid_summary,
        monthly_income=2000
    )
    check_test("Chatbot returns Budget Buddy response", "Budget Buddy Response" in chatbot_response)
    check_test("Chatbot returns a non-empty response", len(chatbot_response.strip()) > 50)
    check_test("Chatbot includes relevant data section", "Relevant data used:" in chatbot_response)
    check_test(
        "Chatbot does not return a hands-on-ai error",
        "❌ Error" not in chatbot_response and "Connection error" not in chatbot_response
    )

    # Test 6: Savings goal calculator works
    savings_result = calculate_savings_goal(5000, 1200, 400)
    check_test("Savings goal calculator returns months needed", "Months needed: 10" in savings_result)

    # Test 7: Savings goal calculator handles invalid inputs
    invalid_savings_result = calculate_savings_goal(5000, 1200, 0)
    check_test("Savings goal handles zero monthly contribution", "Monthly contribution must be greater than zero." in invalid_savings_result)

    # Test 8: Missing file handled safely
    missing_file_result = load_and_clean_transaction_data("missing_file.csv")
    check_test("Missing file returns None", missing_file_result is None)

    # Test 9: Missing required columns handled safely
    missing_columns_csv = io.StringIO(
        "Date,Amount,Description\n"
        "2024-08-01,45.50,Woolworths Weekly Shop\n"
    )
    missing_columns_result = load_and_clean_transaction_data(missing_columns_csv)
    check_test("Missing columns return None", missing_columns_result is None)

    # Test 10: Invalid amount data handled safely
    invalid_amount_csv = io.StringIO(
        "Date,Amount,Category,Description\n"
        "2024-08-01,not_a_number,Groceries,Woolworths Weekly Shop\n"
    )
    invalid_amount_result = load_and_clean_transaction_data(invalid_amount_csv)
    check_test("Invalid amount data returns None", invalid_amount_result is None)

    print("-" * 40)
    print(f"Tests passed: {tests_passed}")
    print(f"Tests failed: {tests_failed}")

    if tests_failed == 0:
        print("All tests passed.")
    else:
        print("Some tests failed. Review the failed cases above.")

run_project_tests()

Budget Buddy Test Results
----------------------------------------
Transaction data loaded and cleaned successfully.
Rows loaded: 25
PASS: Valid CSV loads successfully
PASS: Valid CSV has 25 rows
PASS: Spending summary is created
PASS: Groceries is the largest spending category
PASS: Net expenses are calculated correctly
PASS: Business insights include Budget Buddy heading
PASS: Business insights include groceries
PASS: RAG context includes transaction data heading
PASS: RAG context includes grocery transactions
PASS: Chatbot returns Budget Buddy response
PASS: Chatbot returns Budget Buddy response
PASS: Chatbot returns a non-empty response
PASS: Chatbot includes relevant data section
PASS: Chatbot does not return a hands-on-ai error
PASS: Savings goal calculator returns months needed
PASS: Savings goal handles zero monthly contribution
Error: The transaction file could not be found.
PASS: Missing file returns None
Error: Missing required columns: ['Category']
PASS: Missing columns ret

## Advanced Integration Tests

In [ ]:
# Advanced Integration Tests
# These tests check that the full assistant pipeline works together.

def test_complete_budget_buddy_pipeline():
    """
    Test that the complete Budget Buddy workflow runs from CSV data
    to analysis, RAG context, chatbot output, savings tool, and Gradio backend.
    """
    print("Advanced Integration Test Results")
    print("-" * 40)

    tests_passed = 0
    tests_failed = 0

    def check_test(test_name, condition):
        nonlocal tests_passed, tests_failed

        if condition:
            print(f"PASS: {test_name}")
            tests_passed += 1
        else:
            print(f"FAIL: {test_name}")
            tests_failed += 1

    # Full backend workflow test using the same function as the Gradio UI
    analysis_output, chatbot_output, savings_output = run_budget_buddy_interface(
        uploaded_file=None,
        monthly_income=2000,
        user_question="How can I reduce my grocery spending?",
        goal_amount=5000,
        current_savings=1200,
        monthly_contribution=400
    )

    check_test(
        "Gradio backend returns analysis output",
        "Budget Buddy Finance Insights" in analysis_output
    )

    check_test(
        "Gradio backend returns chatbot output",
        "Budget Buddy Response" in chatbot_output
    )

    check_test(
        "Gradio backend returns savings tool output",
        "Savings Goal Calculator" in savings_output
    )

    check_test(
        "Chatbot uses retrieved grocery context",
        "Woolworths" in chatbot_output and "Groceries" in chatbot_output
    )

    check_test(
        "Savings tool calculates correct time",
        "Months needed: 10" in savings_output
    )

    check_test(
    "Chatbot response does not contain hands-on-ai error",
    "❌ Error" not in chatbot_output and "Connection error" not in chatbot_output
    )
    
    print("-" * 40)
    print(f"Advanced tests passed: {tests_passed}")
    print(f"Advanced tests failed: {tests_failed}")

    if tests_failed == 0:
        print("All advanced integration tests passed.")
    else:
        print("Some advanced integration tests failed. Review the failed cases above.")

test_complete_budget_buddy_pipeline()

Advanced Integration Test Results
----------------------------------------
Transaction data loaded and cleaned successfully.
Rows loaded: 25
PASS: Gradio backend returns analysis output
PASS: Gradio backend returns chatbot output
PASS: Gradio backend returns savings tool output
PASS: Chatbot uses retrieved grocery context
PASS: Savings tool calculates correct time
PASS: Chatbot response does not contain hands-on-ai error
----------------------------------------
Advanced tests passed: 6
Advanced tests failed: 0
All advanced integration tests passed.


## Launch the Gradio App

Run this final cell manually after the tests have passed. It launches the Budget Buddy interface so the user can upload a CSV, enter income, ask a finance question, and use the savings goal tool.

In [82]:
# Launch the Gradio app
# Use share=False for local notebook testing.

budget_buddy_app.launch(share=False)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Transaction data loaded and cleaned successfully.
Rows loaded: 25


---

# 📊 Project Completion Checklist

## Foundation Skills ✅
- [x] **Data Processing**: CSV loading and cleaning functions work reliably
- [x] **Analysis Functions**: Spending summaries calculate correctly
- [x] **Business Insights**: Recommendations are relevant and actionable  
- [x] **Error Handling**: Graceful handling of data issues
- [x] **Testing**: Comprehensive test coverage for core functions
- [x] **Documentation**: Clear AI collaboration documentation in diary

## Advanced Integration ✅
- [x] **Chat Interface**: Finance advisor personality implemented
- [x] **RAG System**: Document retrieval for financial guidance
- [x] **Custom Tools**: At least one financial calculator/utility
- [x] **Gradio UI**: Professional, user-friendly interface
- [x] **Full Integration**: All components work together seamlessly

## Professional Standards ✅
- [x] **Code Quality**: Professional, commented, maintainable code
- [x] **Business Focus**: Clear connection to real finance problems
- [x] **User Experience**: Interface suitable for non-technical users
- [-] **AI Collaboration**: Extensive, well-documented AI usage
- [x] **Testing**: Robust validation of all features

## Project Documentation ✅  
- [x] **Developer's Diary**: Complete AI collaboration documentation
- [x] **README**: Clear project description and usage instructions
- [-] **GitHub**: Regular commits showing development progress
- [x] **Reflection**: Thoughtful analysis of learning and challenges

---

# 🎯 Final Thoughts: Your Finance Assistant Journey

Congratulations on building your Smart Finance Assistant! This project represents a significant achievement in modern business programming:

**Technical Skills Developed:**
- AI-assisted development workflows
- Professional data processing with pandas
- Integration of multiple AI technologies
- User interface design with Gradio
- Comprehensive software testing

**Business Skills Developed:**  
- Financial data analysis and insights
- User-centered application design
- Professional documentation practices
- Iterative development methodology
- Critical evaluation of AI suggestions

**Professional Preparation:**
- Experience with industry-standard AI collaboration
- Portfolio-ready application development
- Understanding of business problem-solving with technology
- Documentation practices for workplace environments

**Your Smart Finance Assistant demonstrates your ability to direct AI tools toward meaningful business solutions - exactly the skill set that modern BIS graduates need for career success!**

---

*Remember to document all AI collaborations in your Developer's Diary and maintain regular GitHub commits throughout your development process.*
